In [9]:
import pandas as pd
import os

In [10]:
artifact_path = "../artifacts/labeled_table.csv"

df = pd.read_csv(artifact_path)

print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

Shape: (96476, 23)

Columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'item_count', 'total_items_price', 'total_freight_value', 'unique_products', 'unique_sellers', 'payment_count', 'total_payment_value', 'payment_types', 'max_installments', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'review_score', 'late']


In [11]:
df["order_purchase_timestamp"] = pd.to_datetime(
    df["order_purchase_timestamp"],
    errors="coerce"
)

print("Minimum purchase date:",
      df["order_purchase_timestamp"].min())

print("Maximum purchase date:",
      df["order_purchase_timestamp"].max())

Minimum purchase date: 2016-09-15 12:16:38
Maximum purchase date: 2018-08-29 15:00:37


In [12]:
monthly_stats = (
    df.groupby(
        df["order_purchase_timestamp"].dt.to_period("M")
    )["late"]
    .agg(["count", "mean"])
)

monthly_stats["late_pct"] = (
    monthly_stats["mean"] * 100
).round(2)

print(monthly_stats.to_string())

                          count      mean  late_pct
order_purchase_timestamp                           
2016-09                       1  1.000000    100.00
2016-10                     270  0.011111      1.11
2016-12                       1  0.000000      0.00
2017-01                     750  0.030667      3.07
2017-02                    1653  0.032063      3.21
2017-03                    2546  0.055774      5.58
2017-04                    2303  0.078593      7.86
2017-05                    3545  0.036107      3.61
2017-06                    3135  0.038596      3.86
2017-07                    3872  0.034349      3.43
2017-08                    4193  0.033150      3.32
2017-09                    4150  0.052048      5.20
2017-10                    4478  0.052925      5.29
2017-11                    7288  0.143112     14.31
2017-12                    5513  0.083802      8.38
2018-01                    7069  0.065639      6.56
2018-02                    6556  0.160006     16.00
2018-03     

In [13]:
df = (
    df.sort_values("order_purchase_timestamp")
      .reset_index(drop=True)
)

print("First order:")
print(
    df[
        ["order_id", "order_purchase_timestamp", "late"]
    ].head(1)
)

print("\nLast order:")
print(
    df[
        ["order_id", "order_purchase_timestamp", "late"]
    ].tail(1)
)

First order:
                           order_id order_purchase_timestamp  late
0  bfbd0f9bdef84302105ad712db648a6c      2016-09-15 12:16:38     1

Last order:
                               order_id order_purchase_timestamp  late
96475  35a972d7f8436f405b56e36add1a7140      2018-08-29 15:00:37     0


In [14]:
print("Overall late rate:")

print(
    df["late"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Overall late rate:
late
0    91.89
1     8.11
Name: proportion, dtype: float64


In [15]:
# 80% Train
# 10% Validation
# 10% Test

n = len(df)

train_end = int(n * 0.80)
val_end = int(n * 0.90)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

Train shape: (77180, 23)
Validation shape: (9648, 23)
Test shape: (9648, 23)


In [16]:
print("=== DATE RANGES ===")

print(
    "Train:",
    train_df["order_purchase_timestamp"].min(),
    "→",
    train_df["order_purchase_timestamp"].max()
)

print(
    "Validation:",
    val_df["order_purchase_timestamp"].min(),
    "→",
    val_df["order_purchase_timestamp"].max()
)

print(
    "Test:",
    test_df["order_purchase_timestamp"].min(),
    "→",
    test_df["order_purchase_timestamp"].max()
)

=== DATE RANGES ===
Train: 2016-09-15 12:16:38 → 2018-05-26 16:54:05
Validation: 2018-05-26 17:57:44 → 2018-07-18 12:59:01
Test: 2018-07-18 12:59:21 → 2018-08-29 15:00:37


In [17]:
print("Train label distribution:")
print(
    train_df["late"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nValidation label distribution:")
print(
    val_df["late"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nTest label distribution:")
print(
    test_df["late"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Train label distribution:
late
0    91.18
1     8.82
Name: proportion, dtype: float64

Validation label distribution:
late
0    97.99
1     2.01
Name: proportion, dtype: float64

Test label distribution:
late
0    91.43
1     8.57
Name: proportion, dtype: float64


In [18]:
for name, part in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    
    monthly = (
        part.groupby(
            part["order_purchase_timestamp"].dt.to_period("M")
        )["late"]
        .agg(["count", "mean"])
    )

    monthly["late_pct"] = (
        monthly["mean"] * 100
    ).round(2)

    print(f"\n{name} monthly late rate:")
    print(monthly.to_string())


Train monthly late rate:
                          count      mean  late_pct
order_purchase_timestamp                           
2016-09                       1  1.000000    100.00
2016-10                     270  0.011111      1.11
2016-12                       1  0.000000      0.00
2017-01                     750  0.030667      3.07
2017-02                    1653  0.032063      3.21
2017-03                    2546  0.055774      5.58
2017-04                    2303  0.078593      7.86
2017-05                    3545  0.036107      3.61
2017-06                    3135  0.038596      3.86
2017-07                    3872  0.034349      3.43
2017-08                    4193  0.033150      3.32
2017-09                    4150  0.052048      5.20
2017-10                    4478  0.052925      5.29
2017-11                    7288  0.143112     14.31
2017-12                    5513  0.083802      8.38
2018-01                    7069  0.065639      6.56
2018-02                    6556  0.160

In [19]:
print("Train unique orders:",
      train_df["order_id"].nunique())

print("Validation unique orders:",
      val_df["order_id"].nunique())

print("Test unique orders:",
      test_df["order_id"].nunique())

train_val_overlap = (
    set(train_df["order_id"])
    & set(val_df["order_id"])
)

train_test_overlap = (
    set(train_df["order_id"])
    & set(test_df["order_id"])
)

val_test_overlap = (
    set(val_df["order_id"])
    & set(test_df["order_id"])
)

print("\nTrain/Validation overlap:",
      len(train_val_overlap))

print("Train/Test overlap:",
      len(train_test_overlap))

print("Validation/Test overlap:",
      len(val_test_overlap))

Train unique orders: 77180
Validation unique orders: 9648
Test unique orders: 9648

Train/Validation overlap: 0
Train/Test overlap: 0
Validation/Test overlap: 0


In [20]:
os.makedirs("../artifacts", exist_ok=True)

train_path = "../artifacts/train.csv"
val_path = "../artifacts/validation.csv"
test_path = "../artifacts/test.csv"

train_df.to_csv(
    train_path,
    index=False
)

val_df.to_csv(
    val_path,
    index=False
)

test_df.to_csv(
    test_path,
    index=False
)

print("Artifacts saved successfully!")

print(train_path)
print(val_path)
print(test_path)

Artifacts saved successfully!
../artifacts/train.csv
../artifacts/validation.csv
../artifacts/test.csv


In [21]:
for path in [
    train_path,
    val_path,
    test_path
]:
    print(
        path,
        "exists:",
        os.path.exists(path)
    )

../artifacts/train.csv exists: True
../artifacts/validation.csv exists: True
../artifacts/test.csv exists: True


In [22]:
train_check = pd.read_csv(train_path)
val_check = pd.read_csv(val_path)
test_check = pd.read_csv(test_path)

print("=== FINAL CHECK ===")

print("\nTrain:", train_check.shape)
print("Validation:", val_check.shape)
print("Test:", test_check.shape)

print("\nMissing labels:")
print(
    "Train:",
    train_check["late"].isna().sum()
)

print(
    "Validation:",
    val_check["late"].isna().sum()
)

print(
    "Test:",
    test_check["late"].isna().sum()
)

print("\nUnique orders:")
print(
    "Train:",
    train_check["order_id"].nunique()
)

print(
    "Validation:",
    val_check["order_id"].nunique()
)

print(
    "Test:",
    test_check["order_id"].nunique()
)

print("\n✅ Notebook 3 completed successfully.")

=== FINAL CHECK ===

Train: (77180, 23)
Validation: (9648, 23)
Test: (9648, 23)

Missing labels:
Train: 0
Validation: 0
Test: 0

Unique orders:
Train: 77180
Validation: 9648
Test: 9648

✅ Notebook 3 completed successfully.
